In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import time

# Configurer le WebDriver pour Chrome
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# URL cible
url = "https://www.vendezvotrevoiture.fr/agences/"
driver.get(url)

# Attendre que la sous-section branches__wrapper soit chargée
try:
    scrollable_section = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, 'branches__wrapper'))
    )
except:
    print("La sous-section 'branches__wrapper' n'a pas été trouvée.")
    driver.quit()
    exit()

# Scroller dans la sous-section pour charger tous les éléments dynamiques
last_height = driver.execute_script("return arguments[0].scrollHeight", scrollable_section)
while True:
    # Scroller jusqu'en bas de la sous-section
    driver.execute_script("arguments[0].scrollTo(0, arguments[0].scrollHeight);", scrollable_section)
    time.sleep(2)  # Attendre le chargement du contenu

    # Vérifier si la hauteur de la sous-section a changé
    new_height = driver.execute_script("return arguments[0].scrollHeight", scrollable_section)
    if new_height == last_height:
        break
    last_height = new_height

# Listes pour stocker les données
garage_names = []
garage_addresses = []

# Rechercher les éléments 'branch__container' dans la sous-section
garages = scrollable_section.find_elements(By.CLASS_NAME, 'branch__container')

for garage in garages:
    # Extraire le nom du garage
    try:
        name = garage.find_element(By.CLASS_NAME, 'branch__name').text
    except:
        name = "N/A"
    
    # Extraire l'adresse du garage
    try:
        # Accéder à la section d'adresse dans la structure imbriquée
        address_section = garage.find_element(By.CLASS_NAME, 'branch__section.section_address')
        address = address_section.find_element(By.CLASS_NAME, 'branch__address').text
    except:
        address = "N/A"
    
    # Ajouter les informations aux listes
    garage_names.append(name)
    garage_addresses.append(address)

# Fermer le navigateur
driver.quit()

# Vérifier si les données ont été collectées
if garage_names and garage_addresses:
    # Créer un DataFrame avec les données extraites
    df = pd.DataFrame({
        'Nom du Garage': garage_names,
        'Adresse': garage_addresses
    })

    # Exporter les données dans un fichier Excel
    df.to_excel('C:/Users/hugoj/Desktop/garages_vendezvotrevoiture_branches_wrapper.xlsx', index=False)
    print("Données extraites et exportées dans 'C:/Users/hugoj/Desktop/garages_vendezvotrevoiture_branches_wrapper.xlsx'")
else:
    print("Aucune donnée n'a été extraite.")


Données extraites et exportées dans 'C:/Users/hugoj/Desktop/garages_vendezvotrevoiture_branches_wrapper.xlsx'
